# Solution E3 · 100% recall with a random forest: the rebalancing done before the split

**Type**: Find the bug

## Objective

Find out why a random forest on a problem with 7% positives gets recall, precision and AUC close to 1: the colleague **duplicated the positives to balance the classes before** splitting training and test, so
the same examples are in both. Fix it (split first, rebalancing only on the training set) and measure the true performance.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split


def generate():
    X, y = make_classification(n_samples=3000, n_features=10, n_informative=4, n_redundant=0, weights=[0.94], class_sep=0.8, flip_y=0.02, random_state=0)
    df = pd.DataFrame(X, columns=[f"x{i}" for i in range(10)])
    df["y"] = y
    return df


df = generate()
rf = lambda **kw: RandomForestClassifier(200, random_state=0, n_jobs=-1, **kw)
print("fraction of positives:", df["y"].mean().round(3))

## Commented solution

In [ ]:
majority, minority = df[df["y"] == 0], df[df["y"] == 1]
balanced = pd.concat([majority, minority.sample(len(majority), replace=True, random_state=0)], ignore_index=True)     # 50% / 50%
Xa_bug, Xt_bug, ya_bug, yt_bug = train_test_split(balanced.drop(columns="y"), balanced["y"], test_size=0.3, stratify=balanced["y"], random_state=0)
m_bug = rf().fit(Xa_bug, ya_bug)
pred_bug, score_bug = m_bug.predict(Xt_bug), m_bug.predict_proba(Xt_bug)[:, 1]
bug_recall, bug_precision, bug_auc = recall_score(yt_bug, pred_bug), precision_score(yt_bug, pred_bug), roc_auc_score(yt_bug, score_bug)
print(f"recall {bug_recall:.3f} | precision {bug_precision:.3f} | AUC {bug_auc:.3f}")

**Where the bug is.** The positives (about 200) were duplicated until they equaled the negatives (about 2,800): every positive appears on average 13-14 times. Then the random split puts some copies in the training set and others in the test set. For the model,
recognizing a positive of the test set means **remembering an identical copy** seen in training: a random forest memorizes them.

In [ ]:
test_duplicates_share = Xt_bug[yt_bug == 1].merge(Xa_bug.drop_duplicates(), how="left", indicator=True)["_merge"].eq("both").mean()
print(f"positives of the test set that appear identical in the training set: {test_duplicates_share:.1%}")

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df.drop(columns="y"), df["y"], test_size=0.3, stratify=df["y"], random_state=0)
tr = pd.concat([X_tr, y_tr], axis=1)
maj, mino = tr[tr["y"] == 0], tr[tr["y"] == 1]
tr_balanced = pd.concat([maj, mino.sample(len(maj), replace=True, random_state=0)], ignore_index=True)       # the duplication happens ONLY in the training set
m_ok = rf().fit(tr_balanced.drop(columns="y"), tr_balanced["y"])
pred_ok, score_ok = m_ok.predict(X_te), m_ok.predict_proba(X_te)[:, 1]
ok_recall, ok_precision = recall_score(y_te, pred_ok), precision_score(y_te, pred_ok)
ok_auc, ok_ap = roc_auc_score(y_te, score_ok), average_precision_score(y_te, score_ok)
print(f"original test set ({y_te.mean():.1%} positives): recall {ok_recall:.3f} | precision {ok_precision:.3f} | AUC {ok_auc:.3f} | AP {ok_ap:.3f}")

In [ ]:
m_weights = rf(class_weight="balanced").fit(X_tr, y_tr)
weights_ap = average_precision_score(y_te, m_weights.predict_proba(X_te)[:, 1])
weights_recall = recall_score(y_te, m_weights.predict(X_te))
print(f"class_weight='balanced': AP {weights_ap:.3f} | recall {weights_recall:.3f}")

## Check

In [ ]:
with verify():
    check(bug_recall > 0.95 and bug_precision > 0.95 and bug_auc > 0.98, "the colleague's numbers are too good to be true", "don't modify the colleague's code")
    check(test_duplicates_share is not None and test_duplicates_share > 0.9, "over 90% of the positives of the test set appear identical in the training set", "merge between Xt_bug (positives only) and Xa_bug.drop_duplicates()")
    check(ok_recall < 0.4 and 0.75 < ok_auc < 0.92, "on the true test set: low recall (below 0.4) and AUC 0.75-0.92", "the test set must have the original prevalence (7%) and no duplicates")
    check(ok_ap < 0.6, "average precision below 0.6: the problem is hard", "average_precision_score on the original test set")
    check(weights_ap is not None and abs(weights_ap - ok_ap) < 0.15 and weights_recall < 0.4, "class_weight='balanced' gives comparable results, with no duplicates and no leakage", "RandomForestClassifier(class_weight='balanced') on the original training set")
done()

## What you learned

- With the rebalancing **before** the split every positive of the test set has an identical copy in the training set: the random forest remembers it and gets recall 1.0, precision 0.99 and AUC 1.0. On the true test set, with the
  original prevalence, the recall is 0.21 and the AUC 0.83. A test set with copies of the training set measures nothing.
- Rule: **split first, always.** Oversampling, SMOTE, augmentation, imputation, scaling, feature selection: anything that looks at the data and modifies it goes after the split and only on the training set (in a `Pipeline` or with `imblearn.pipeline.Pipeline`
  if you use samplers). And the test set keeps the **real prevalence**.
- Duplicated data does not move the problem: with the correct rebalancing the random forest reaches an average precision of 0.42, practically that of a model with `class_weight` (0.415). Rebalancing changes the working point (more recall, less precision), it does not add information.
- Warning sign: a model with **all** the metrics close to 1 on a problem with rare classes and noise (`flip_y`) is almost always leakage. To check it, look for duplicated rows between training and test.

## Common pitfall

Balancing the dataset "to prepare it" in a data-cleaning notebook and then doing the split in the modeling notebook. The balanced dataset must never exist outside a training pipeline.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why must SMOTE be applied after the split and inside the cross-validation?</summary>

SMOTE creates synthetic examples by interpolating between nearby positives: if I apply it before the split, a synthetic example in the training set is almost identical to the real positive that ends up in the test set, so the model "recognizes" it. You need
a pipeline that applies the sampling only on the training part of every fold (with `imblearn.pipeline.Pipeline`), and the test/validation must keep the real distribution.

</details>